In [ ]:
!mkdir -p data/raws/oews

In [ ]:
!mkdir -p data/raw/eloundou

In [ ]:
!ls data/raws/oews

all_data_M_2019.xlsx  all_data_M_2022.xlsx  all_data_M_2025.xlsx
all_data_M_2020.xlsx  all_data_M_2023.xlsx
all_data_M_2021.xlsx  all_data_M_2024.xlsx


In [ ]:
"""
Process OEWS data - takes excel/zip for ea yr as input
"""
import sys
import logging
import numpy as np
import pandas as pd
from pathlib import Path

RAW_DIR = Path("data/raw/oews")
PROCESSED_DIR = Path("data/processed")

YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
TREATMENT_YEAR = 2022 # gpt launched nov 2022

# All BLS suppression flags (only '*' appears at national/cross-industry level
# in practice, but we guard against the full documented set)
SUPPRESSION_FLAGS = {"*", "**", "#", "~", "-", "–", "N/A", "NA"}

In [ ]:
SOC_CROSSWALK_2019 = {
    # 2019 code            new code    note
    "15-1256": ("15-1252", "Software Developers: 2019 bundled QA analysts; "
                            "2020+ split. 2019 wage/emp slightly broader."),
    "15-2098": ("15-2051", "Data Scientists: 2019 was residual 'all other' category; "
                            "2020+ became standalone. 2019 wage/emp slightly broader."),
    "13-2098": ("13-2051", "Financial Analysts: 2019 bundled risk specialists & "
                            "others; 2020+ standalone. 2019 wage/emp slightly broader."),
}

# focal occupations
FOCAL_OCCS = {
    "15-1252": "Software Developers",
    "15-2051": "Data Scientists",
    "13-2051": "Financial Analysts",
    "23-2011": "Paralegals and Legal Assistants",
    "35-3023": "Fast Food and Counter Workers",
    "47-2061": "Construction Laborers",
    "31-1120": "Home Health and Personal Care Aides",
}


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# helper fns

def to_numeric_safe(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    """
    Convert a string Series that may contain BLS suppression flags to float.

    Returns
    -------
    numeric    : float Series  (suppressed / unparseable → NaN)
    suppressed : bool Series   (True where original value was a flag)
    """
    s = series.astype(str).str.strip()
    suppressed = s.isin(SUPPRESSION_FLAGS) | s.isin({"nan", ""})
    s_clean = s.copy()
    s_clean[suppressed] = np.nan
    s_clean = s_clean.str.replace(",", "", regex=False)   # remove thousands commas
    numeric = pd.to_numeric(s_clean, errors="coerce")
    return numeric, suppressed


def apply_soc_crosswalk(df: pd.DataFrame, year: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    For the 2019 file, recode legacy SOC 2010 codes to SOC 2018 equivalents.

    Returns
    -------
    df            : DataFrame with occ_code updated
    crosswalk_log : DataFrame recording every row that was remapped (for audit)
    """
    df["soc_remapped"] = 0
    remapped_rows = []

    if year != 2019:
        return df, pd.DataFrame()

    for old_code, (new_code, note) in SOC_CROSSWALK_2019.items():
        mask = df["occ_code"] == old_code
        if mask.any():
            original_title = df.loc[mask, "occ_title"].iloc[0]
            df.loc[mask, "occ_code"]    = new_code
            df.loc[mask, "soc_remapped"] = 1
            remapped_rows.append({
                "year":           year,
                "old_soc_code":   old_code,
                "new_soc_code":   new_code,
                "original_title": original_title,
                "note":           note,
            })
            log.info(
                f"  SOC crosswalk: {old_code} → {new_code}  "
                f"({original_title})"
            )
        else:
            log.warning(
                f"  SOC crosswalk: {old_code} not found in 2019 data. "
                "This is unexpected — check the file."
            )

    crosswalk_log = pd.DataFrame(remapped_rows)
    return df, crosswalk_log


def read_one_year(year: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load, filter, and clean OEWS data for a single survey year.

    Applies three filters (area / o_group / i_group), converts numerics,
    flags suppression, applies SOC crosswalk for 2019, adds derived columns.

    Returns (data_df, crosswalk_log_df).
    Both are empty DataFrames if the file is missing.
    """
    filepath = RAW_DIR / f"all_data_M_{year}.xlsx"

    if not filepath.exists():
        log.error(
            f"File not found: {filepath}\n"
            f"  Download: https://www.bls.gov/oes/special.requests/"
            f"oesm{str(year)[2:]}nat.zip\n"
            f"  Extract all_data_M_{year}.xlsx  →  {RAW_DIR}/"
        )
        return pd.DataFrame(), pd.DataFrame()

    log.info(f"  Reading {filepath.name} ...")

    # dtype=str preserves suppression flags ('*', '#', etc.)
    raw = pd.read_excel(filepath, dtype=str, sheet_name=0)
    raw.columns = raw.columns.str.strip().str.lower()

    # column val
    required = {"area", "o_group", "i_group", "occ_code", "occ_title",
                "tot_emp", "a_mean", "h_mean"}
    missing_cols = required - set(raw.columns)
    if missing_cols:
        log.error(
            f"  Year {year}: missing columns {missing_cols}. "
            f"Columns found: {list(raw.columns)}"
        )
        return pd.DataFrame(), pd.DataFrame()

    total_rows = len(raw)

    # three filters
    mask = (
        (raw["area"].str.strip()    == "99")             &  # U.S. national
        (raw["o_group"].str.strip() == "detailed")       &  # 6-digit SOC only
        (raw["i_group"].str.strip() == "cross-industry")    # all-industry agg
    )
    df = raw[mask].copy()

    log.info(
        f"  {total_rows:>7,} raw rows  →  {len(df):,} "
        f"national × detailed × cross-industry rows"
    )

    if df.empty:
        log.error(
            f"  Year {year}: 0 rows after filtering. "
            "Confirm the xlsx is the all_data (national) file, not state/metro."
        )
        return pd.DataFrame(), pd.DataFrame()

    # standardize cols strings
    df["occ_code"]  = df["occ_code"].str.strip()
    df["occ_title"] = df["occ_title"].str.strip()

    # SOC vintage crosswalk (2019 only)
    df, crosswalk_log = apply_soc_crosswalk(df, year)

    # conver numerics + flag suppression
    df["a_mean"],  df["suppressed_wage"] = to_numeric_safe(df["a_mean"])
    df["tot_emp"], _                     = to_numeric_safe(df["tot_emp"])
    df["h_mean"],  _                     = to_numeric_safe(df["h_mean"])

    n_supp = int(df["suppressed_wage"].sum())
    if n_supp > 0:
        suppressed_titles = df.loc[df["suppressed_wage"], "occ_title"].tolist()
        log.info(
            f"  Suppressed a_mean: {n_supp} rows (kept with NaN): "
            f"{suppressed_titles}"
        )

    # annual_only flag
    # BLS 'annual' column = TRUE when the occupation reports no hourly wage
    if "annual" in df.columns:
        df["annual_only"] = (
            df["annual"].str.strip().str.upper() == "TRUE"
        ).astype(int)
    else:
        df["annual_only"] = 0

    # outcomes
    df["wage_bill"] = df["a_mean"] * df["tot_emp"]
    df["log_wage"]  = np.log(df["a_mean"])    # NaN propagates for suppressed rows
    df["log_emp"]   = np.log(df["tot_emp"])
    df["log_wbill"] = np.log(df["wage_bill"])

    df["year"]       = year
    df["post"]       = int(year > TREATMENT_YEAR)
    df["event_time"] = year - TREATMENT_YEAR   # τ: −3,−2,−1,0,+1,+2

    # output cols
    keep = [
        "year", "event_time", "post",
        "occ_code", "occ_title",
        "tot_emp", "a_mean", "h_mean",
        "wage_bill", "log_wage", "log_emp", "log_wbill",
        "annual_only", "suppressed_wage", "soc_remapped",
    ]
    return df[keep].reset_index(drop=True), crosswalk_log


# main pipleine:

def build_panel() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Stack all years, add the balanced-panel flag, split out audit logs.

    Returns (panel, suppression_log, crosswalk_log).
    """
    frames, crosswalk_frames = [], []

    for year in YEARS:
        df, cw_log = read_one_year(year)
        if not df.empty:
            frames.append(df)
        if not cw_log.empty:
            crosswalk_frames.append(cw_log)

    if not frames:
        log.error(
            f"No data loaded. Place OEWS xlsx files in {RAW_DIR}/ and retry."
        )
        sys.exit(1)

    panel  = pd.concat(frames, ignore_index=True)
    n_years = panel["year"].nunique()
    log.info(
        f"\nStacked: {len(panel):,} rows across "
        f"{n_years} years {sorted(panel['year'].unique())}"
    )

    # ── Balanced-panel flag ───────────────────────────────────────────────────
    year_counts   = panel.groupby("occ_code")["year"].count()
    balanced_occs = year_counts[year_counts == n_years].index
    panel["balanced"] = panel["occ_code"].isin(balanced_occs).astype(int)

    n_bal   = panel.loc[panel["balanced"] == 1, "occ_code"].nunique()
    n_total = panel["occ_code"].nunique()
    log.info(
        f"Balanced (all {n_years} years): {n_bal:,} of {n_total:,} occupations"
    )

    suppression_log = (
        panel[panel["suppressed_wage"] == 1]
        .copy()
        .sort_values(["occ_code", "year"])
    )
    crosswalk_log = (
        pd.concat(crosswalk_frames, ignore_index=True)
        if crosswalk_frames else pd.DataFrame()
    )

    panel = panel.sort_values(["occ_code", "year"]).reset_index(drop=True)
    return panel, suppression_log, crosswalk_log




def print_sanity_checks(panel: pd.DataFrame) -> None:
    """Printed diagnostics to verify the panel before saving."""

    sep = "=" * 68
    log.info(f"\n{sep}\nSANITY CHECKS\n{sep}")

    # 1. Shape
    log.info(
        f"\nPanel shape      : {panel.shape}"
        f"\nYears            : {sorted(panel['year'].unique())}"
        f"\nTotal occupations: {panel['occ_code'].nunique():,}"
        f"\nBalanced occs    : "
        f"{panel.loc[panel['balanced']==1,'occ_code'].nunique():,}"
        f"\nSOC-remapped rows: {int(panel['soc_remapped'].sum())} (2019 only)"
    )

    # 2. Event-time distribution
    log.info("\nEvent-time (τ) distribution:")
    ev = panel.groupby("event_time").agg(
        survey_year   = ("year",       "first"),
        n_occupations = ("occ_code",   "count"),
        post          = ("post",        "first"),
    )
    print(ev.to_string())

    # 3. National aggregates by year
    log.info("\nNational aggregates by year (eyeball for plausibility):")
    agg = panel.groupby("year").agg(
        mean_wage  = ("a_mean",          "mean"),
        total_emp  = ("tot_emp",         "sum"),
        n_occ      = ("occ_code",        "count"),
        pct_supp   = ("suppressed_wage", "mean"),
    )
    agg["mean_wage"] = agg["mean_wage"].map("${:>10,.0f}".format)
    agg["total_emp"] = agg["total_emp"].map("{:>14,.0f}".format)
    agg["pct_supp"]  = (agg["pct_supp"] * 100).map("{:.1f}%".format)
    print(agg.to_string())

    # 4. Focal occupations — confirm all seven are present across all years
    log.info("\nFocal occupation check (all 7 from proposal):")
    for soc, title in FOCAL_OCCS.items():
        rows = panel[panel["occ_code"] == soc][
            ["year", "a_mean", "tot_emp", "log_wage", "event_time", "soc_remapped"]
        ]
        if rows.empty:
            log.warning(f"  ✗  {soc} ({title}) — NOT FOUND.")
        else:
            flag = "  [includes 2019 crosswalk row]" if rows["soc_remapped"].any() else ""
            print(f"\n  ✓  {soc} — {title}{flag}")
            print(rows.drop(columns="soc_remapped").to_string(index=False))

    # 5. Log-outcome distributions
    log.info("\nLog-outcome descriptive stats (non-suppressed rows):")
    clean = panel[panel["suppressed_wage"] == 0]
    print(
        clean[["log_wage", "log_emp", "log_wbill"]]
        .describe()
        .round(3)
        .to_string()
    )



In [ ]:
log.info("=" * 68)
log.info("OEWS NATIONAL PANEL — PROCESSING")
log.info("=" * 68)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

panel, suppression_log, crosswalk_log = build_panel()
print_sanity_checks(panel)

# ── Save outputs ──────────────────────────────────────────────────────────
out_panel    = PROCESSED_DIR / "oews_national_panel.csv"
out_supp     = PROCESSED_DIR / "oews_suppression_log.csv"
out_crosswalk= PROCESSED_DIR / "oews_crosswalk_log.csv"

panel.to_csv(out_panel, index=False)
suppression_log.to_csv(out_supp, index=False)
if not crosswalk_log.empty:
    crosswalk_log.to_csv(out_crosswalk, index=False)

log.info("\nOutputs saved:")
log.info(f"  {out_panel}      ({len(panel):,} rows)")
log.info(f"  {out_supp}   ({len(suppression_log):,} rows)")
if not crosswalk_log.empty:
    log.info(f"  {out_crosswalk}   ({len(crosswalk_log):,} remapped codes)")

log.info("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Column guide for Person C (estimation scripts)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
occ_code         SOC 2018 code  ← MERGE KEY for script 02 (Eloundou)
year             OEWS survey reference year
log_wage         ln(a_mean)     ← OUTCOME 1
log_emp          ln(tot_emp)    ← OUTCOME 2
log_wbill        ln(wage_bill)  ← OUTCOME 3  (composition-bias check)
post             1 = year > 2022
event_time       τ = year − 2022  (event-study dummies for Method 4)
balanced         1 = present in ALL years (restrict for Methods 3 & 4)
suppressed_wage  1 = a_mean withheld by BLS (exclude from regressions)
soc_remapped     1 = 2019 SOC code reassigned to SOC 2018 equivalent
                    (flag as limitation in the Data section)

Next step → run 02_process_eloundou.py to merge LLM exposure scores.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

            survey_year  n_occupations  post
event_time                                  
-3                 2019            789     0
-2                 2020            789     0
-1                 2021            831     0
 0                 2022            830     0
 1                 2023            831     1
 2                 2024            831     1
 3                 2025            830     1
        mean_wage       total_emp  n_occ pct_supp
year                                             
2019  $    62,329     146,873,470    789     0.5%
2020  $    64,170     139,096,790    789     0.5%
2021  $    68,496     140,885,570    831     0.7%
2022  $    72,172     147,884,760    830     0.6%
2023  $    75,628     151,853,170    831     0.6%
2024  $    78,130     154,186,300    831     0.6%
2025  $    80,229     155,481,840    830     0.6%

  ✓  15-1252 — Software Developers  [includes 2019 crosswalk row]
 year   a_mean   tot_emp  log_wage  event_time
 2019 111620.0 1406870.0 11.622

In [ ]:
"""
merge OEWS Panel with Eloundou et al. LLM Exposure Scores

INPUTS
------
  data/processed/oews_national_panel.csv
  data/raw/eloundou/occ_level.csv
"""
import logging
import numpy as np
import pandas as pd
from pathlib import Path


PROCESSED_DIR = Path("data/processed")
RAW_ELOUNDOU  = Path("data/raw/eloundou/occ_level.csv")

# Primary and robustness exposure measures
EXPOSURE_COLS = [
    "dv_alpha", "dv_beta", "dv_gamma",
    "human_alpha", "human_beta", "human_gamma",
]

# "All Other" and residual category patterns — these get a special flag
RESIDUAL_PATTERNS = [
    "all other", "not elsewhere classified", "n.e.c.",
    "miscellaneous", "except ", ", all other",
]


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# load and aggregate Eloundou exposure scores

def load_exposure(path: Path) -> pd.DataFrame:
    """
    Load the Eloundou occ_level.csv file, truncate O*NET codes to 6-digit
    SOC codes, and aggregate subspecialties by taking the mean within each SOC.

    Returns one row per SOC code with all 6 exposure measures.
    """
    if not path.exists():
        raise FileNotFoundError(
            f"Eloundou file not found: {path}\n"
            "Download from: https://github.com/openai/evals or the paper's\n"
            "supplementary materials at https://arxiv.org/abs/2303.10130\n"
            f"Place occ_level.csv in: {path.parent}/"
        )

    raw = pd.read_csv(path, dtype=str)
    log.info(f"Eloundou file: {len(raw):,} O*NET rows, columns: {list(raw.columns)}")

    # Rename for clarity
    raw = raw.rename(columns={
        "O*NET-SOC Code":    "onet_code",
        "Title":             "onet_title",
        "dv_rating_alpha":   "dv_alpha",
        "dv_rating_beta":    "dv_beta",
        "dv_rating_gamma":   "dv_gamma",
        "human_rating_alpha":"human_alpha",
        "human_rating_beta": "human_beta",
        "human_rating_gamma":"human_gamma",
    })

    # Truncate O*NET code "11-1011.00" → SOC code "11-1011"
    raw["soc_code"] = raw["onet_code"].str[:7]

    # Convert exposure columns to numeric
    for col in EXPOSURE_COLS:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")

    # Aggregate: mean across subspecialties within each SOC code
    # (follows Eloundou et al.'s own aggregation convention)
    agg = raw.groupby("soc_code").agg(
        **{col: (col, "mean") for col in EXPOSURE_COLS},
        n_onet=("onet_code", "count"),
    ).reset_index()

    n_multi = (agg["n_onet"] > 1).sum()
    log.info(
        f"Aggregated to {len(agg):,} SOC codes "
        f"({n_multi} had multiple O*NET subspecialties → averaged)"
    )
    return agg


# sibling-based imputation for unmatched OEWS codes

def build_sibling_imputer(expo_agg: pd.DataFrame) -> dict[str, dict]:
    """
    Build a lookup: for any 7-char SOC code prefix, return the mean exposure
    across all Eloundou codes sharing the first 5 characters.

    Used to impute exposure for OEWS codes not directly in Eloundou.
    e.g., 31-1120 → mean of 31-1121 and 31-1122
    """
    expo_agg["prefix5"] = expo_agg["soc_code"].str[:5]
    sibling_lookup = {}
    for prefix, group in expo_agg.groupby("prefix5"):
        sibling_lookup[prefix] = {
            col: group[col].mean() for col in EXPOSURE_COLS
        }
        sibling_lookup[prefix]["n_onet"]    = len(group)
        sibling_lookup[prefix]["siblings"]  = group["soc_code"].tolist()
    return sibling_lookup


def is_residual(title: str) -> bool:
    """Return True if the occupation title looks like a residual 'All Other' category."""
    t = str(title).lower()
    return any(pat in t for pat in RESIDUAL_PATTERNS)


# merge and impute

def merge_and_impute(
    panel: pd.DataFrame,
    expo_agg: pd.DataFrame,
    sibling_lookup: dict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Left-merge OEWS panel with exposure scores.
    For unmatched codes, attempt sibling-based imputation.

    Returns (merged_panel, diagnostics_df).
    """
    n_panel = len(panel)

    # Direct merge
    merged = panel.merge(
        expo_agg.rename(columns={"soc_code": "occ_code"}),
        on="occ_code",
        how="left",
        validate="m:1",    # many panel rows per SOC code (one per year)
    )

    assert len(merged) == n_panel, "Merge changed row count — check for duplicates"

    # Initialise imputation flags
    merged["exposure_imputed"]  = 0
    merged["exposure_residual"] = 0

    # Identify unmatched occupations (missing dv_beta after direct merge)
    unmatched_occs = (
        merged[merged["dv_beta"].isna()]["occ_code"]
        .unique()
    )
    log.info(
        f"\nDirect merge: {panel['occ_code'].nunique() - len(unmatched_occs):,} "
        f"SOC codes matched directly, {len(unmatched_occs):,} unmatched"
    )

    diagnostics = []

    # Use the most recent year's title for residual detection.
    # Earlier years may carry legacy "All Other" titles for codes that were
    # later renamed
    latest_title = (
        merged.sort_values("year")
        .groupby("occ_code")["occ_title"]
        .last()
    )

    for occ in unmatched_occs:
        prefix5 = occ[:5]
        title   = latest_title.get(occ, merged.loc[merged["occ_code"] == occ, "occ_title"].iloc[0])
        residual_flag = int(is_residual(title))

        if prefix5 in sibling_lookup:
            sib = sibling_lookup[prefix5]
            # Fill exposure columns from sibling mean
            for col in EXPOSURE_COLS:
                merged.loc[merged["occ_code"] == occ, col] = sib[col]
            merged.loc[merged["occ_code"] == occ, "n_onet"]            = sib["n_onet"]
            merged.loc[merged["occ_code"] == occ, "exposure_imputed"]  = 1
            merged.loc[merged["occ_code"] == occ, "exposure_residual"] = residual_flag

            diagnostics.append({
                "occ_code":         occ,
                "occ_title":        title,
                "match_status":     "sibling_imputed",
                "imputed_from":     str(sib["siblings"]),
                "dv_beta_imputed":  round(sib["dv_beta"], 4),
                "exposure_residual":residual_flag,
            })
            log.info(
                f"  Imputed {occ} ({title[:40]}) "
                f"← siblings {sib['siblings']} → dv_beta={sib['dv_beta']:.3f}"
            )
        else:
            diagnostics.append({
                "occ_code":         occ,
                "occ_title":        title,
                "match_status":     "unmatched_no_siblings",
                "imputed_from":     "",
                "dv_beta_imputed":  np.nan,
                "exposure_residual":residual_flag,
            })
            log.warning(
                f"  Unmatched (no siblings): {occ} ({title[:40]}) "
                "— will have NaN exposure"
            )

    # Also log the directly matched codes
    matched_occs = set(panel["occ_code"].unique()) - set(unmatched_occs)
    for occ in sorted(matched_occs):
        title = latest_title.get(occ, panel.loc[panel["occ_code"] == occ, "occ_title"].iloc[0])
        dv_b  = expo_agg.loc[expo_agg["soc_code"] == occ, "dv_beta"]
        diagnostics.append({
            "occ_code":         occ,
            "occ_title":        title,
            "match_status":     "direct",
            "imputed_from":     "",
            "dv_beta_imputed":  round(float(dv_b.iloc[0]), 4) if len(dv_b) else np.nan,
            "exposure_residual":int(is_residual(title)),
        })

    diagnostics_df = pd.DataFrame(diagnostics).sort_values(
        ["match_status", "occ_code"]
    ).reset_index(drop=True)

    return merged, diagnostics_df



def print_sanity_checks(merged: pd.DataFrame, diag: pd.DataFrame) -> None:
    sep = "=" * 68
    log.info(f"\n{sep}\nSANITY CHECKS\n{sep}")

    # 1. Match rate
    total_occs   = merged["occ_code"].nunique()
    matched      = diag[diag["match_status"] == "direct"]["occ_code"].nunique()
    imputed      = diag[diag["match_status"] == "sibling_imputed"]["occ_code"].nunique()
    unmatched    = diag[diag["match_status"] == "unmatched_no_siblings"]["occ_code"].nunique()
    null_expo    = merged["dv_beta"].isna().sum()

    log.info(
        f"\nCoverage across {total_occs:,} OEWS occupations:"
        f"\n  Direct match       : {matched:,} ({100*matched/total_occs:.1f}%)"
        f"\n  Sibling imputed    : {imputed:,} ({100*imputed/total_occs:.1f}%)"
        f"\n  Unmatched (NaN)    : {unmatched:,} ({100*unmatched/total_occs:.1f}%)"
        f"\n  Residual categories: "
        f"{diag['exposure_residual'].sum()} (flagged, exclude from main regressions)"
        f"\n  Panel rows with NaN exposure: {null_expo:,} of {len(merged):,}"
    )

    # 2. Exposure distribution — full sample vs. balanced panel only
    log.info("\nExposure (dv_beta) distribution — unique occupations, one value each:")
    occ_expo = (
        merged[merged["year"] == merged["year"].min()]
        [["occ_code", "dv_beta", "exposure_imputed", "balanced"]]
        .dropna(subset=["dv_beta"])
    )
    print(occ_expo["dv_beta"].describe().round(3).to_string())

    # 3. Focal occupations
    focal = {
        "15-1252": "Software Developers",
        "15-2051": "Data Scientists",
        "13-2051": "Financial Analysts",
        "23-2011": "Paralegals and Legal Assistants",
        "35-3023": "Fast Food and Counter Workers",
        "47-2061": "Construction Laborers",
        "31-1120": "Home Health and Personal Care Aides",
    }
    log.info("\nFocal occupation exposure scores:")
    cols = ["occ_code", "occ_title", "dv_beta", "human_beta",
            "exposure_imputed", "n_onet"]
    one_year = merged[merged["year"] == merged["year"].min()]
    for soc, label in focal.items():
        row = one_year[one_year["occ_code"] == soc]
        if row.empty:
            log.warning(f"  ✗  {soc} ({label}) — NOT in merged panel")
        else:
            r = row.iloc[0]
            imputed_note = " [IMPUTED]" if r["exposure_imputed"] else ""
            print(
                f"  ✓  {soc} {label:40s}"
                f"  dv_beta={r['dv_beta']:.3f}"
                f"  human_beta={r['human_beta']:.3f}"
                f"{imputed_note}"
            )

    # 4. High vs low exposure: does the ranking match prior expectations?
    log.info(
        "\nTop 10 highest dv_beta occupations (should be high-knowledge roles):"
    )
    top10 = (
        occ_expo.sort_values("dv_beta", ascending=False)
        .head(10)[["occ_code", "dv_beta"]]
        .merge(
            merged[["occ_code", "occ_title"]].drop_duplicates("occ_code"),
            on="occ_code"
        )
    )
    print(top10[["occ_code", "occ_title", "dv_beta"]].to_string(index=False))

    log.info(
        "\nTop 10 lowest dv_beta occupations (should be manual/physical roles):"
    )
    bot10 = (
        occ_expo.sort_values("dv_beta", ascending=True)
        .head(10)[["occ_code", "dv_beta"]]
        .merge(
            merged[["occ_code", "occ_title"]].drop_duplicates("occ_code"),
            on="occ_code"
        )
    )
    print(bot10[["occ_code", "occ_title", "dv_beta"]].to_string(index=False))

    # 5. dv_beta vs human_beta correlation (measure consistency check)
    corr = merged[["dv_beta", "human_beta"]].dropna().corr().iloc[0, 1]
    log.info(
        f"\nCorrelation dv_beta vs human_beta: {corr:.3f} "
        f"(expect ~0.5–0.7; both measure same concept via different raters)"
    )

    # 6. Reminder about residual categories
    residual_occs = diag[diag["exposure_residual"] == 1]["occ_code"].tolist()
    log.info(
        f"\nResidual 'All Other' categories ({len(residual_occs)} occupations):"
        f"\n  Recommend excluding from main regressions — add 'exposure_residual==0'"
        f"\n  filter in your estimation script."
        f"\n  These codes: {residual_occs[:10]}{'...' if len(residual_occs)>10 else ''}"
    )


def main() -> None:
    log.info("=" * 68)
    log.info("MERGE: OEWS PANEL × ELOUNDOU EXPOSURE SCORES")
    log.info("=" * 68)

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

    # Load OEWS panel
    oews_path = PROCESSED_DIR / "oews_national_panel.csv"
    if not oews_path.exists():
        raise FileNotFoundError(
            f"{oews_path} not found. Run 01_process_oews.py first."
        )
    panel = pd.read_csv(oews_path)
    log.info(f"OEWS panel: {len(panel):,} rows, {panel['occ_code'].nunique():,} occupations")

    # Load and aggregate exposure scores
    expo_agg = load_exposure(RAW_ELOUNDOU)

    # Build sibling imputer
    sibling_lookup = build_sibling_imputer(expo_agg)

    # Merge + impute
    merged, diagnostics = merge_and_impute(panel, expo_agg, sibling_lookup)

    # Sanity checks
    print_sanity_checks(merged, diagnostics)

    # Save
    out_merged = PROCESSED_DIR / "oews_exposure_merged.csv"
    out_diag   = PROCESSED_DIR / "merge_diagnostics.csv"

    merged.to_csv(out_merged, index=False)
    diagnostics.to_csv(out_diag, index=False)

    log.info(f"\nOutputs saved:")
    log.info(f"  {out_merged}  ({len(merged):,} rows)")
    log.info(f"  {out_diag}   ({len(diagnostics):,} occupation-level match records)")

    log.info("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Column guide for Person C (estimation scripts)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTCOMES (from OEWS):
  log_wage, log_emp, log_wbill    three regression outcomes

TREATMENT (from Eloundou):
  dv_beta          PRIMARY exposure variable  [0,1]
  human_beta       Robustness check exposure  [0,1]

TIMING:
  post             1 = year > 2022  (post-ChatGPT)
  event_time       τ = year − 2022  (event-study period dummies)

FIXED EFFECTS:
  occ_code         occupation FE
  year             year FE  (or interact with controls for two-way FE)

SAMPLE RESTRICTIONS (apply in estimation):
  balanced == 1           balanced panel (required for Methods 3 & 4)
  suppressed_wage == 0    drop BLS-suppressed wage observations
  exposure_residual == 0  drop residual "All Other" categories
  dv_beta.notna()         drop occupations with no exposure score

DiD INTERACTION TERM:
  dv_beta × post          this is your β in the OLS DiD (Method 1)

Next step → run 03_process_onet_controls.py to add O*NET control variables.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    """)


if __name__ == "__main__":
    main()


count    789.000
mean       0.320
std        0.219
min        0.000
25%        0.115
50%        0.315
75%        0.488
max        1.000
  ✓  15-1252 Software Developers                       dv_beta=0.868  human_beta=0.447
  ✓  15-2051 Data Scientists                           dv_beta=0.695  human_beta=0.582
  ✓  13-2051 Financial Analysts                        dv_beta=0.462  human_beta=0.500
  ✓  23-2011 Paralegals and Legal Assistants           dv_beta=0.525  human_beta=0.450
  ✓  35-3023 Fast Food and Counter Workers             dv_beta=0.097  human_beta=0.115
  ✓  47-2061 Construction Laborers                     dv_beta=0.050  human_beta=0.025
  ✓  31-1120 Home Health and Personal Care Aides       dv_beta=0.105  human_beta=0.136 [IMPUTED]
occ_code                                            occ_title  dv_beta
 15-2021                                       Mathematicians 1.000000
 43-9081                        Proofreaders and Copy Markers 0.975000
 43-4021                        

In [ ]:
!zip -r data_backup.zip data/processed/

  adding: data/processed/ (stored 0%)
  adding: data/processed/merge_diagnostics.csv (deflated 74%)
  adding: data/processed/oews_exposure_merged.csv (deflated 78%)
  adding: data/processed/oews_crosswalk_log.csv (deflated 52%)
  adding: data/processed/oews_national_panel.csv (deflated 68%)
  adding: data/processed/oews_suppression_log.csv (deflated 68%)


In [ ]:
!mkdir -p data/raw/onet

In [ ]:
"""
O*NET Controls for double lasso

inputs:
  data/raw/onet/Education_Training_and_Experience.xlsx
  data/raw/onet/Work_Activities.xlsx
  data/processed/oews_exposure_merged.csv   ← from 02_process_eloundou.py

outputs:
   data/processed/onet_controls.csv          ← one row per SOC code
  data/processed/oews_exposure_onet.csv     ← full merged panel
"""

import logging
import numpy as np
import pandas as pd
from pathlib import Path


RAW_ONET      = Path("data/raw/onet")
PROCESSED_DIR = Path("data/processed")

EDUCATION_FILE   = RAW_ONET / "Education_Training_and_Experience.xlsx"
WORK_ACT_FILE    = RAW_ONET / "Work_Activities.xlsx"
PANEL_INPUT      = PROCESSED_DIR / "oews_exposure_merged.csv"
CONTROLS_OUT     = PROCESSED_DIR / "onet_controls.csv"
MERGED_OUT       = PROCESSED_DIR / "oews_exposure_onet.csv"

# Education categories that count as bachelor's degree or higher
# BACHELORS_PLUS_CATEGORIES = {8, 9, 10, 11, 12}
BACHELORS_PLUS_CATEGORIES = {6, 7, 8, 9, 10, 11, 12}

# Work activity element IDs by task type
ROUTINE_COGNITIVE_ELEMENTS = {
    "4.A.2.a.2",   # Processing Information
    "4.A.4.c.1",   # Performing Administrative Activities
    "4.A.3.b.6",   # Documenting/Recording Information
}
ROUTINE_MANUAL_ELEMENTS = {
    "4.A.3.a.3",   # Controlling Machines and Processes
    "4.A.3.a.2",   # Handling and Moving Objects
    "4.A.3.a.1",   # Performing General Physical Activities
}
NONROUTINE_ANALYTIC_ELEMENTS = {
    "4.A.2.a.4",   # Analyzing Data or Information
    "4.A.2.b.1",   # Making Decisions and Solving Problems
    "4.A.2.b.2",   # Thinking Creatively
}
NONROUTINE_INTERPERSONAL_ELEMENTS = {
    "4.A.4.a.4",   # Establishing and Maintaining Interpersonal Relationships
    "4.A.4.a.5",   # Assisting and Caring for Others
    "4.A.4.a.3",   # Communicating with People Outside the Organization
}

ALL_TASK_ELEMENTS = (
    ROUTINE_COGNITIVE_ELEMENTS
    | ROUTINE_MANUAL_ELEMENTS
    | NONROUTINE_ANALYTIC_ELEMENTS
    | NONROUTINE_INTERPERSONAL_ELEMENTS
)

# Focal occupations for sanity checks
FOCAL_OCCS = {
    "15-1252": "Software Developers",
    "15-2051": "Data Scientists",
    "13-2051": "Financial Analysts",
    "23-2011": "Paralegals and Legal Assistants",
    "35-3023": "Fast Food and Counter Workers",
    "47-2061": "Construction Laborers",
    "31-1120": "Home Health and Personal Care Aides",
}


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# helpers

def onet_to_soc(onet_code: str) -> str:
    """Truncate O*NET code '11-1011.00' → SOC code '11-1011'."""
    return str(onet_code).strip()[:7]


def standardize(series: pd.Series) -> pd.Series:
    """Z-score standardize a series (mean 0, SD 1), ignoring NaN."""
    return (series - series.mean()) / series.std()


def build_education_controls(path: Path) -> pd.DataFrame:
    """
    From the Education, Training, and Experience file, construct:
      pct_bachelors_plus : % of workers requiring bachelor's degree or higher

    Returns one row per SOC code.
    """
    log.info(f"Reading education file: {path.name}")
    raw = pd.read_excel(path, dtype=str)
    raw.columns = raw.columns.str.strip().str.lower().str.replace(" ", "_")

    # Rename O*NET code column (handles slight naming differences)
    onet_col = [c for c in raw.columns if "o*net" in c or "onet" in c][0]
    raw = raw.rename(columns={onet_col: "onet_code"})

    raw["soc_code"] = raw["onet_code"].apply(onet_to_soc)

    # Filter to Required Level of Education (element 2.D.1, scale RL)
    edu = raw[
        (raw["element_id"].str.strip() == "2.D.1") &
        (raw["scale_id"].str.strip()   == "RL")
    ].copy()

    edu["category"]   = pd.to_numeric(edu["category"],   errors="coerce")
    edu["data_value"] = pd.to_numeric(edu["data_value"], errors="coerce")

    log.info(f"  Education rows (element 2.D.1, scale RL): {len(edu):,}")

    # Flag bachelor's+ categories
    edu["is_bachelors_plus"] = edu["category"].isin(BACHELORS_PLUS_CATEGORIES)

    # For each O*NET code: sum Data Value for bachelor's+ categories
    onet_edu = (
        edu.groupby("onet_code")
        .apply(lambda g: g.loc[g["is_bachelors_plus"], "data_value"].sum())
        .reset_index()
        .rename(columns={0: "pct_bachelors_plus"})
    )
    onet_edu["soc_code"] = onet_edu["onet_code"].apply(onet_to_soc)

    # Aggregate to SOC level (mean across subspecialties)
    soc_edu = (
        onet_edu.groupby("soc_code")["pct_bachelors_plus"]
        .mean()
        .reset_index()
    )

    log.info(
        f"  Education controls: {len(soc_edu):,} SOC codes\n"
        f"  pct_bachelors_plus range: "
        f"{soc_edu['pct_bachelors_plus'].min():.1f} – "
        f"{soc_edu['pct_bachelors_plus'].max():.1f}"
    )

    siblings = {
        "13-2051": ["13-2052", "13-2053"],
        "31-1120": ["31-1121", "31-1122"],
    }
    edu_cols = ["pct_bachelors_plus"]
    for soc, sibs in siblings.items():
        sib_rows = soc_edu[soc_edu["soc_code"].isin(sibs)]
        if not sib_rows.empty:
            imputed = sib_rows[edu_cols].mean()
            imputed["soc_code"] = soc
            soc_edu = pd.concat(
                [soc_edu, pd.DataFrame([imputed])], ignore_index=True
            )
            log.info(f"  Imputed {soc} from siblings {sibs}")
    return soc_edu


# RTI controls

def build_work_activity_controls(path: Path) -> pd.DataFrame:
    """
    From the Work Activities file, construct four task indices and the
    composite RTI score. Uses Importance (IM) scale only.

    Returns one row per SOC code with columns:
      routine_cognitive_index, routine_manual_index,
      nonroutine_analytic_index, nonroutine_interpersonal_index,
      routine_task_intensity
    """
    log.info(f"Reading work activities file: {path.name}")
    raw = pd.read_excel(path, dtype=str)
    raw.columns = raw.columns.str.strip().str.lower().str.replace(" ", "_")

    onet_col = [c for c in raw.columns if "o*net" in c or "onet" in c][0]
    raw = raw.rename(columns={onet_col: "onet_code"})

    raw["soc_code"]    = raw["onet_code"].apply(onet_to_soc)
    raw["element_id"]  = raw["element_id"].str.strip()
    raw["scale_id"]    = raw["scale_id"].str.strip()
    raw["data_value"]  = pd.to_numeric(raw["data_value"], errors="coerce")

    # Keep only Importance scale and relevant elements
    wa = raw[
        (raw["scale_id"]   == "IM") &
        (raw["element_id"].isin(ALL_TASK_ELEMENTS))
    ].copy()

    log.info(f"  Work activity rows (IM scale, target elements): {len(wa):,}")

    # Tag each row by task type
    def tag_task_type(elem_id):
        if elem_id in ROUTINE_COGNITIVE_ELEMENTS:      return "rc"
        if elem_id in ROUTINE_MANUAL_ELEMENTS:         return "rm"
        if elem_id in NONROUTINE_ANALYTIC_ELEMENTS:    return "nra"
        if elem_id in NONROUTINE_INTERPERSONAL_ELEMENTS: return "nri"
        return None

    wa["task_type"] = wa["element_id"].apply(tag_task_type)

    # Mean importance per O*NET code × task type
    onet_task = (
        wa.groupby(["onet_code", "task_type"])["data_value"]
        .mean()
        .unstack("task_type")
        .reset_index()
    )
    onet_task["soc_code"] = onet_task["onet_code"].apply(onet_to_soc)

    # Aggregate to SOC code (mean across subspecialties)
    soc_task = (
        onet_task.groupby("soc_code")[["rc", "rm", "nra", "nri"]]
        .mean()
        .reset_index()
        .rename(columns={
            "rc":  "routine_cognitive_index",
            "rm":  "routine_manual_index",
            "nra": "nonroutine_analytic_index",
            "nri": "nonroutine_interpersonal_index",
        })
    )

    # Composite RTI = (RC + RM) - (NRA + NRI), standardized
    soc_task["rti_raw"] = (
        (soc_task["routine_cognitive_index"] + soc_task["routine_manual_index"])
        - (soc_task["nonroutine_analytic_index"] + soc_task["nonroutine_interpersonal_index"])
    )
    soc_task["routine_task_intensity"] = standardize(soc_task["rti_raw"])
    soc_task = soc_task.drop(columns="rti_raw")

    log.info(
        f"  Task controls: {len(soc_task):,} SOC codes\n"
        f"  RTI range: "
        f"{soc_task['routine_task_intensity'].min():.2f} – "
        f"{soc_task['routine_task_intensity'].max():.2f} (standardized)"
    )

    # Sibling imputation for SOC codes not directly in O*NET
    siblings = {
        "13-2051": ["13-2052", "13-2053"],
        "31-1120": ["31-1121", "31-1122"],
    }
    task_cols = [
        "routine_cognitive_index", "routine_manual_index",
        "nonroutine_analytic_index", "nonroutine_interpersonal_index",
        "routine_task_intensity",
    ]
    for soc, sibs in siblings.items():
        sib_rows = soc_task[soc_task["soc_code"].isin(sibs)]
        if not sib_rows.empty:
            imputed = sib_rows[task_cols].mean()
            imputed["soc_code"] = soc
            soc_task = pd.concat(
                [soc_task, pd.DataFrame([imputed])], ignore_index=True
            )
            log.info(f"  Imputed {soc} from siblings {sibs}")
    return soc_task


# merge controls

def build_onet_controls(edu_df: pd.DataFrame, task_df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge education and task controls into one occupation-level table.
    Uses outer merge so we keep occupations that appear in one file but not the other.
    """
    controls = edu_df.merge(task_df, on="soc_code", how="outer")
    log.info(
        f"Combined O*NET controls: {len(controls):,} SOC codes "
        f"({controls['pct_bachelors_plus'].notna().sum()} with education, "
        f"{controls['routine_task_intensity'].notna().sum()} with RTI)"
    )
    return controls


def merge_onto_panel(
    panel: pd.DataFrame,
    controls: pd.DataFrame,
) -> pd.DataFrame:
    """
    Left-merge O*NET controls onto the OEWS × Eloundou panel.
    Controls are occupation-level (time-invariant), so each occupation
    gets the same control values across all years — this is correct.
    """
    n_before = len(panel)

    merged = panel.merge(
        controls.rename(columns={"soc_code": "occ_code"}),
        on="occ_code",
        how="left",
        validate="m:1",
    )

    assert len(merged) == n_before, "Row count changed during merge — check for duplicates"

    # Flag whether controls were matched
    merged["onet_controls_matched"] = merged["pct_bachelors_plus"].notna().astype(int)

    n_matched = merged.loc[merged["year"] == merged["year"].min(), "onet_controls_matched"].sum()
    n_total   = merged["occ_code"].nunique()
    log.info(
        f"Panel merge: {n_matched:,} of {n_total:,} occupations have O*NET controls "
        f"({100*n_matched/n_total:.1f}%)"
    )
    return merged


def print_sanity_checks(merged: pd.DataFrame, controls: pd.DataFrame) -> None:
    sep = "=" * 68
    log.info(f"\n{sep}\nSANITY CHECKS\n{sep}")

    # 1. Control variable distributions
    log.info("\nControl variable distributions (across occupations):")
    ctrl_cols = [
        "pct_bachelors_plus",
        "routine_cognitive_index",
        "routine_manual_index",
        "nonroutine_analytic_index",
        "nonroutine_interpersonal_index",
        "routine_task_intensity",
    ]
    one_year = merged[merged["year"] == merged["year"].min()].drop_duplicates("occ_code")
    print(one_year[ctrl_cols].describe().round(3).to_string())

    # 2. Focal occupation check
    log.info("\nFocal occupations — expect high education + low RTI for knowledge workers:")
    display_cols = [
        "occ_code", "pct_bachelors_plus",
        "routine_cognitive_index", "routine_manual_index",
        "nonroutine_analytic_index", "routine_task_intensity",
    ]
    for soc, label in FOCAL_OCCS.items():
        row = one_year[one_year["occ_code"] == soc]
        if row.empty:
            log.warning(f"  ✗  {soc} ({label}) — NOT in merged panel")
        else:
            r = row.iloc[0]
            print(
                f"\n  ✓  {soc} {label}"
                f"\n     pct_bachelors_plus={r['pct_bachelors_plus']:.1f}%"
                f"  RTI={r['routine_task_intensity']:.2f}"
                f"  analytic={r['nonroutine_analytic_index']:.2f}"
                f"  manual={r['routine_manual_index']:.2f}"
            )

    # 3. RTI vs LLM exposure correlation (should be negative)
    corr = one_year[["routine_task_intensity", "dv_beta"]].dropna().corr().iloc[0, 1]
    log.info(
        f"\nCorrelation RTI × dv_beta (LLM exposure): {corr:.3f}"
        f"\n  Expect negative: high-RTI jobs are routine → lower LLM exposure"
        f"\n  (If positive, check element ID mapping above)"
    )

    # 4. Education vs LLM exposure correlation (should be positive)
    corr_edu = one_year[["pct_bachelors_plus", "dv_beta"]].dropna().corr().iloc[0, 1]
    log.info(
        f"\nCorrelation pct_bachelors_plus × dv_beta: {corr_edu:.3f}"
        f"\n  Expect positive: high-education jobs have more LLM-exposed tasks"
    )

    # 5. Coverage
    n_missing = (one_year["onet_controls_matched"] == 0).sum()
    log.info(
        f"\nOccupations missing O*NET controls: {n_missing:,}"
        f"\n  These will have NaN controls and drop out of Double Lasso — "
        f"check if any are focal occupations (should be 0)."
    )


def main() -> None:
    log.info("=" * 68)
    log.info("O*NET CONTROLS — PROCESSING AND MERGE")
    log.info("=" * 68)

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

    # Check inputs exist
    for f in [EDUCATION_FILE, WORK_ACT_FILE, PANEL_INPUT]:
        if not f.exists():
            raise FileNotFoundError(
                f"Missing input: {f}\n"
                "Place O*NET files in data/raw/onet/ and run scripts 01 and 02 first."
            )

    # Build controls
    edu_controls  = build_education_controls(EDUCATION_FILE)
    task_controls = build_work_activity_controls(WORK_ACT_FILE)
    controls      = build_onet_controls(edu_controls, task_controls)

    # Save occupation-level controls table
    controls.to_csv(CONTROLS_OUT, index=False)
    log.info(f"\nSaved: {CONTROLS_OUT}  ({len(controls):,} occupation rows)")

    # Load panel and merge
    log.info(f"\nLoading panel: {PANEL_INPUT}")
    panel  = pd.read_csv(PANEL_INPUT)
    merged = merge_onto_panel(panel, controls)

    # Sanity checks
    print_sanity_checks(merged, controls)

    # Save merged panel
    merged.to_csv(MERGED_OUT, index=False)
    log.info(f"\nSaved: {MERGED_OUT}  ({len(merged):,} rows)")

    log.info("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Column guide for your Double Lasso script
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTCOMES:         log_wage, log_emp, log_wbill
TREATMENT:        dv_beta  (× post for DiD interaction)
TIMING:           post, event_time
FIXED EFFECTS:    occ_code, year

W VECTOR (Double Lasso controls — all pre-determined, time-invariant):
  pct_bachelors_plus             education level
  routine_cognitive_index        routine cognitive task content
  routine_manual_index           routine manual task content
  nonroutine_analytic_index      non-routine analytic content
  nonroutine_interpersonal_index interpersonal task content
  routine_task_intensity         RTI composite (standardized)

SAMPLE FILTERS (apply before Double Lasso):
  balanced == 1
  suppressed_wage == 0
  exposure_residual == 0
  dv_beta.notna()
  onet_controls_matched == 1

Next: implement Double Lasso DiD using this panel.
See Belloni, Chernozhukov & Hansen (2014) and course Lectures 3-5.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    """)


if __name__ == "__main__":
    main()

/tmp/ipykernel_3619/292985538.py:262: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.loc[g["is_bachelors_plus"], "data_value"].sum())


       pct_bachelors_plus  routine_cognitive_index  routine_manual_index  nonroutine_analytic_index  nonroutine_interpersonal_index  routine_task_intensity
count             703.000                  711.000               711.000                    711.000                         711.000                 711.000
mean               35.581                    3.300                 2.913                      3.537                           3.257                   0.033
std                40.122                    0.551                 0.878                      0.527                           0.545                   1.003
min                 0.000                    1.613                 1.067                      1.877                           1.847                  -2.405
25%                 0.000                    2.918                 2.152                      3.243                           2.868                  -0.772
50%                13.120                    3.393              

In [ ]:
edu = pd.read_excel("data/raw/onet/Education_Training_and_Experience.xlsx", dtype=str)
edu.columns = edu.columns.str.strip().str.lower().str.replace(" ", "_")
onet_col = [c for c in edu.columns if "o*net" in c or "onet" in c][0]
print(edu[(edu[onet_col].str.startswith("15-1252")) & (edu["element_id"].str.strip() == "2.D.1")][["category","data_value"]].to_string())

     category data_value
4584        1          0
4585        2       3.13
4586        3       1.67
4587        4          0
4588        5       5.07
4589        6       84.8
4590        7       0.81
4591        8       4.52
4592        9          0
4593       10          0
4594       11          0
4595       12          0


In [ ]:
raw = pd.read_excel("data/raw/onet/Work_Activities.xlsx", dtype=str)
onet_col = [c for c in raw.columns if "o*net" in c.lower() or "onet" in c.lower()][0]
codes = raw[onet_col].str.strip().unique()
# Check siblings
for prefix in ["13-205", "31-112"]:
    matches = [c for c in codes if c.startswith(prefix)]
    print(prefix, "→", matches)

13-205 → ['13-2052.00', '13-2053.00']
31-112 → ['31-1121.00', '31-1122.00']


# Double Lasso

In [6]:
import logging
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional

from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

warnings.filterwarnings("ignore", category=FutureWarning)


PROCESSED_DIR = Path("data/processed")
RESULTS_DIR   = Path("results")

INPUT_FILE = PROCESSED_DIR / "oews_exposure_onet.csv"

# Outcomes to run Double Lasso for
OUTCOMES = {
    "log_wage":  "Log mean wage (primary)",
    "log_emp":   "Log employment",
    "log_wbill": "Log wage bill (composition-bias check)",
}

# Treatment variable and timing
TREATMENT_VAR = "dv_beta"
POST_VAR      = "post"
OCC_VAR       = "occ_code"
YEAR_VAR      = "year"

# W vector: pre-determined occupation-level controls
CONTROL_VARS = [
    "pct_bachelors_plus",
    "routine_cognitive_index",
    "routine_manual_index",
    "nonroutine_analytic_index",
    "nonroutine_interpersonal_index",
    "routine_task_intensity",
]

# LassoCV settings
CV_FOLDS    = 5
MAX_ITER    = 10_000
RANDOM_SEED = 42

# Whether to also compute occupation-clustered SEs (slower but more conservative)
CLUSTER_SE = True


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)



def load_and_filter(path: Path) -> pd.DataFrame:
    """Load panel and apply sample restrictions."""
    df = pd.read_csv(path)
    log.info(f"Loaded: {len(df):,} rows, {df['occ_code'].nunique():,} occupations")

    before = df["occ_code"].nunique()

    df = df[
        (df["balanced"]              == 1) &
        (df["suppressed_wage"]       == 0) &
        (df["exposure_residual"]     == 0) &
        (df["dv_beta"].notna())            &
        (df["onet_controls_matched"] == 1)
    ].copy()

    after = df["occ_code"].nunique()
    log.info(
        f"After sample restrictions: {len(df):,} rows, {after:,} occupations "
        f"({before - after} dropped)"
    )
    log.info(
        f"Years: {sorted(df[YEAR_VAR].unique())}\n"
        f"Pre-period: {df[df[POST_VAR]==0][YEAR_VAR].nunique()} years  "
        f"Post-period: {df[df[POST_VAR]==1][YEAR_VAR].nunique()} years"
    )
    return df


def add_did_treatment(df: pd.DataFrame) -> pd.DataFrame:
    """
    Construct the DiD interaction: D = dv_beta × post.
    This is a continuous-treatment DiD: β captures the effect of a
    1-unit increase in LLM exposure on the outcome in the post period.
    """
    df["D"] = df[TREATMENT_VAR] * df[POST_VAR]
    log.info(
        f"DiD interaction D = dv_beta × post:\n"
        f"  mean={df['D'].mean():.3f}  std={df['D'].std():.3f}  "
        f"  min={df['D'].min():.3f}  max={df['D'].max():.3f}"
    )
    return df


# within-demean - partial out fixed effects

def within_demean(
    df: pd.DataFrame,
    cols: list[str],
) -> pd.DataFrame:
    """
    Remove occupation and year fixed effects by within-group demeaning.

    For a two-way FE panel, we use the iterative demeaning approach:
      1. Subtract occupation means
      2. Subtract year means (of the occupation-demeaned variable)
      Repeat until convergence (typically 2-3 iterations suffice).

    Returns a DataFrame of demeaned columns (same index as df).
    """
    demeaned = df[cols].copy().astype(float)

    for _ in range(10):   # iterate to convergence
        prev = demeaned.copy()

        # Subtract occupation means
        occ_means = demeaned.groupby(df[OCC_VAR]).transform("mean")
        demeaned  = demeaned - occ_means

        # Subtract year means
        yr_means  = demeaned.groupby(df[YEAR_VAR]).transform("mean")
        demeaned  = demeaned - yr_means

        # Check convergence
        max_change = (demeaned - prev).abs().max().max()
        if max_change < 1e-10:
            break

    return demeaned


def run_lasso_cv(X: np.ndarray, y: np.ndarray, label: str) -> tuple[np.ndarray, np.ndarray, float]:
    """
    Fit LassoCV, return residuals, selected variable mask, and chosen alpha.
    X should already be demeaned; we standardize within this function.
    """
    scaler  = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    lasso = LassoCV(
        cv          = CV_FOLDS,
        max_iter    = MAX_ITER,
        random_state= RANDOM_SEED,
        n_jobs      = -1,
    )
    lasso.fit(X_scaled, y)

    selected = lasso.coef_ != 0
    residuals = y - lasso.predict(X_scaled)

    log.info(
        f"  Lasso ({label}): alpha={lasso.alpha_:.6f}, "
        f"selected {selected.sum()} of {X.shape[1]} controls"
    )
    return residuals, selected, lasso.alpha_


def double_lasso_did(
    df: pd.DataFrame,
    outcome: str,
    control_vars: list[str],
) -> dict:
    """
    Run Double Lasso DiD for one outcome variable.

    Returns a dict with the point estimate, SE, t-stat, p-value,
    selected controls, and diagnostics.
    """
    log.info(f"\n{'─'*60}")
    log.info(f"Double Lasso DiD — outcome: {outcome}")
    log.info(f"{'─'*60}")

    # Drop rows missing the outcome
    sub = df.dropna(subset=[outcome] + control_vars + ["D"]).copy()
    log.info(f"  Sample: {len(sub):,} obs, {sub[OCC_VAR].nunique():,} occupations")

    # Columns to demean
    all_cols = [outcome, "D"] + control_vars
    demeaned = within_demean(sub, all_cols)

    Y_tilde = demeaned[outcome].values
    D_tilde = demeaned["D"].values
    W_tilde = demeaned[control_vars].values

    # Step 1: Lasso of Ỹ on W̃ → residuals ṽ
    v_tilde, selected_y, alpha_y = run_lasso_cv(W_tilde, Y_tilde, f"outcome ({outcome})")

    # Step 2: Lasso of D̃ on W̃ → residuals ũ
    u_tilde, selected_d, alpha_d = run_lasso_cv(W_tilde, D_tilde, "treatment (D)")

    # Step 3: OLS of ṽ on ũ → β̂
    u_with_const = sm.add_constant(u_tilde)
    ols = sm.OLS(v_tilde, u_with_const).fit(cov_type="HC3")

    beta    = ols.params[1]
    se_hc3  = ols.bse[1]
    tstat   = ols.tvalues[1]
    pval    = ols.pvalues[1]
    ci_low  = ols.conf_int()[1, 0]
    ci_high = ols.conf_int()[1, 1]
    # ci_low  = ols.conf_int().iloc[1, 0]
    # ci_high = ols.conf_int().iloc[1, 1]

    # Clustered SE (by occupation) — more conservative
    se_clustered = None
    if CLUSTER_SE:
        groups = sub[OCC_VAR].values
        ols_cl = sm.OLS(v_tilde, u_with_const).fit(
            cov_type="cluster",
            cov_kwds={"groups": groups},
        )
        se_clustered = ols_cl.bse[1]
        log.info(
            f"  β̂ = {beta:.4f}  SE(HC3)={se_hc3:.4f}  "
            f"SE(clustered)={se_clustered:.4f}  p={pval:.4f}"
        )
    else:
        log.info(f"  β̂ = {beta:.4f}  SE={se_hc3:.4f}  p={pval:.4f}")

    # Which controls were selected?
    selected_names = [
        v for v, s_y, s_d in zip(control_vars, selected_y, selected_d)
        if s_y or s_d
    ]
    log.info(f"  Controls selected (in either equation): {selected_names}")

    return {
        "outcome":         outcome,
        "beta":            round(beta, 6),
        "se_hc3":          round(se_hc3, 6),
        "se_clustered":    round(se_clustered, 6) if se_clustered is not None else None,
        "tstat":           round(tstat, 4),
        "pval":            round(pval, 4),
        "ci_low_95":       round(ci_low, 6),
        "ci_high_95":      round(ci_high, 6),
        "n_obs":           len(sub),
        "n_occ":           sub[OCC_VAR].nunique(),
        "n_years":         sub[YEAR_VAR].nunique(),
        "alpha_y":         round(alpha_y, 8),
        "alpha_d":         round(alpha_d, 8),
        "selected_in_y":   [v for v, s in zip(control_vars, selected_y) if s],
        "selected_in_d":   [v for v, s in zip(control_vars, selected_d) if s],
        "selected_either": selected_names,
    }


def format_results_table(results: list[dict]) -> pd.DataFrame:
    """Produce a clean results table for the paper."""
    rows = []
    for r in results:
        sig = ""
        if r["pval"] < 0.01:  sig = "***"
        elif r["pval"] < 0.05: sig = "**"
        elif r["pval"] < 0.10: sig = "*"

        rows.append({
            "Outcome":          r["outcome"],
            "β (DiD)":          f"{r['beta']:.4f}{sig}",
            "SE (HC3)":         f"({r['se_hc3']:.4f})",
            "SE (clustered)":   f"({r['se_clustered']:.4f})" if r["se_clustered"] else "",
            "p-value":          f"{r['pval']:.4f}",
            "95% CI":           f"[{r['ci_low_95']:.4f}, {r['ci_high_95']:.4f}]",
            "N obs":            r["n_obs"],
            "N occupations":    r["n_occ"],
            "Controls selected":str(r["selected_either"]),
        })
    return pd.DataFrame(rows)


def print_results_table(results: list[dict]) -> None:
    sep = "=" * 68
    log.info(f"\n{sep}\nDOUBLE LASSO DiD — RESULTS SUMMARY\n{sep}")
    log.info("Outcome: log_wage / log_emp / log_wbill")
    log.info("Treatment: dv_beta × post (continuous LLM exposure × post-ChatGPT)")
    log.info("Fixed effects: occupation + year (within-demeaned)")
    log.info("Controls: O*NET education + task indices (Double Lasso selected)")
    log.info("* p<0.10  ** p<0.05  *** p<0.01  (HC3 robust SEs)\n")

    for r in results:
        sig = ""
        if r["pval"] < 0.01:   sig = "***"
        elif r["pval"] < 0.05: sig = "**"
        elif r["pval"] < 0.10: sig = "*"

        interp = interpret_beta(r["outcome"], r["beta"])

        print(
            f"  {r['outcome']:12s}  β={r['beta']:+.4f}{sig:3s}  "
            f"SE={r['se_hc3']:.4f}  p={r['pval']:.4f}  "
            f"CI=[{r['ci_low_95']:.4f}, {r['ci_high_95']:.4f}]"
        )
        print(f"             → {interp}")
        if r["se_clustered"]:
            print(f"             SE(clustered by occ)={r['se_clustered']:.4f}")
        print()


def interpret_beta(outcome: str, beta: float) -> str:
    """Plain-English interpretation of the DiD coefficient."""
    direction = "increase" if beta > 0 else "decrease"
    pct = abs(beta) * 100

    if outcome == "log_wage":
        return (
            f"A 1-unit increase in LLM exposure is associated with a "
            f"{pct:.1f}% {direction} in mean wages post-ChatGPT, "
            f"relative to low-exposure occupations. "
            f"{'Consistent with augmentation.' if beta > 0 else 'Consistent with displacement.'}"
        )
    elif outcome == "log_emp":
        return (
            f"A 1-unit increase in LLM exposure is associated with a "
            f"{pct:.1f}% {direction} in employment post-ChatGPT. "
            f"{'Consistent with augmentation (more workers hired).' if beta > 0 else 'Consistent with displacement (fewer workers).'}"
        )
    elif outcome == "log_wbill":
        return (
            f"Wage bill {direction}s by {pct:.1f}%. "
            f"Compare to wage and employment results to assess composition bias."
        )
    return ""



In [ ]:
def main() -> None:
    log.info("=" * 68)
    log.info("METHOD 3: DOUBLE LASSO DiD")
    log.info("=" * 68)

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"{INPUT_FILE} not found. Run 03_process_onet_controls.py first."
        )

    # Load and prepare data
    df = load_and_filter(INPUT_FILE)
    df = add_did_treatment(df)

    # Confirm control vars are present
    missing_controls = [c for c in CONTROL_VARS if c not in df.columns]
    if missing_controls:
        raise ValueError(f"Missing control columns: {missing_controls}")

    # Run Double Lasso for each outcome
    all_results = []
    for outcome, label in OUTCOMES.items():
        log.info(f"\nOutcome: {label}")
        if df[outcome].notna().sum() == 0:
            log.warning(f"  No non-null observations for {outcome} — skipping.")
            continue
        result = double_lasso_did(df, outcome, CONTROL_VARS)
        all_results.append(result)

    # Print summary
    print_results_table(all_results)

    # Save results
    results_df = format_results_table(all_results)
    results_df.to_csv(RESULTS_DIR / "double_lasso_results.csv", index=False)

    # Save selected variables detail
    selected_df = pd.DataFrame([
        {
            "outcome":       r["outcome"],
            "selected_in_outcome_eq": str(r["selected_in_y"]),
            "selected_in_treatment_eq": str(r["selected_in_d"]),
            "selected_either": str(r["selected_either"]),
            "lasso_alpha_outcome":   r["alpha_y"],
            "lasso_alpha_treatment": r["alpha_d"],
        }
        for r in all_results
    ])
    selected_df.to_csv(RESULTS_DIR / "double_lasso_selected_vars.csv", index=False)

    log.info(f"\nOutputs saved to {RESULTS_DIR}/")
    log.info("  double_lasso_results.csv")
    log.info("  double_lasso_selected_vars.csv")

    log.info("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INTERPRETATION GUIDE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
β > 0 on log_wage  → augmentation (LLM exposure raises wages)
β < 0 on log_wage  → displacement (LLM exposure lowers wages)
β > 0 on log_emp   → augmentation (more workers hired in exposed occs)
β < 0 on log_emp   → displacement (fewer workers in exposed occs)

Compare wage bill (log_wbill) to log_wage:
  If log_wage ↑ but log_wbill ≈ 0 → composition bias likely
  (displacement of low-wage workers mechanically raises average wage)

Null result (β ≈ 0, large p): consistent with Acemoglu (2024) —
  early ChatGPT may have been too limited to show equilibrium effects
  in the 2022-2024 window. Frame as "no detectable effect yet."

Compare your β to Method 1 (OLS DiD) and Method 2 (Shift-Share IV):
  Double Lasso β closer to IV than OLS → OLS was confounded by
  occupation-level sorting. Double Lasso successfully conditions on it.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    """)


if __name__ == "__main__":
    main()

  log_wage      β=-0.0748***  SE=0.0057  p=0.0000  CI=[-0.0860, -0.0636]
             → A 1-unit increase in LLM exposure is associated with a 7.5% decrease in mean wages post-ChatGPT, relative to low-exposure occupations. Consistent with displacement.
             SE(clustered by occ)=0.0084

  log_emp       β=+0.0668***  SE=0.0165  p=0.0001  CI=[0.0344, 0.0991]
             → A 1-unit increase in LLM exposure is associated with a 6.7% increase in employment post-ChatGPT. Consistent with augmentation (more workers hired).
             SE(clustered by occ)=0.0318

  log_wbill     β=-0.0076     SE=0.0176  p=0.6648  CI=[-0.0421, 0.0269]
             → Wage bill decreases by 0.8%. Compare to wage and employment results to assess composition bias.
             SE(clustered by occ)=0.0328



In [ ]:
!zip -r project_outputs.zip data/processed/ results/

  adding: data/processed/ (stored 0%)
  adding: data/processed/merge_diagnostics.csv (deflated 74%)
  adding: data/processed/oews_exposure_merged.csv (deflated 78%)
  adding: data/processed/oews_crosswalk_log.csv (deflated 52%)
  adding: data/processed/oews_national_panel.csv (deflated 68%)
  adding: data/processed/onet_controls.csv (deflated 73%)
  adding: data/processed/oews_suppression_log.csv (deflated 68%)
  adding: data/processed/oews_exposure_onet.csv (deflated 81%)
  adding: results/ (stored 0%)
  adding: results/double_lasso_selected_vars.csv (deflated 76%)
  adding: results/double_lasso_results.csv (deflated 55%)


## Adding ACS

In [ ]:
import pandas as pd
acs = pd.read_csv("usa_00001.csv.gz", compression="gzip")

In [ ]:
print(acs.shape)
print(acs.head(3))
print(acs.columns.tolist())

(16694227, 17)
   YEAR  SAMPLE  SERIAL       CBSERIAL  HHWT        CLUSTER  STRATA  GQ  \
0  2019  201901       1  2019010000088  11.0  2019000000011  220001   4   
1  2019  201901       2  2019010000096  70.0  2019000000021  100001   3   
2  2019  201901       3  2019010000153  20.0  2019000000031  110001   4   

   PERNUM  PERWT  SEX  AGE  RACE  RACED  EDUC  EDUCD   OCC  
0       1   11.0    1   39     2    200     4     40     0  
1       1   70.0    2   21     1    100     4     40     0  
2       1   20.0    1   19     2    200     7     71  5240  
['YEAR', 'SAMPLE', 'SERIAL', 'CBSERIAL', 'HHWT', 'CLUSTER', 'STRATA', 'GQ', 'PERNUM', 'PERWT', 'SEX', 'AGE', 'RACE', 'RACED', 'EDUC', 'EDUCD', 'OCC']


In [ ]:
print(acs["OCC"].value_counts().head(10))
print(acs["EDUCD"].unique())

OCC
0       6759002
440      275782
9130     220493
4720     209963
3255     208012
2310     200466
4760     198869
5240     169319
4700     164841
9620     159243
Name: count, dtype: int64
[ 40  71  30  65  63 101  64  50 114  81  14   2  17  26  25  23  16  61
  15 116  22 115  12   1  11]


In [ ]:
print(acs[acs["OCC"] > 0]["OCC"].min(), acs[acs["OCC"] > 0]["OCC"].max())

10 9920


In [ ]:
import pandas as pd

In [ ]:
acs = pd.read_csv("usa_00002.csv.gz", compression="gzip",
                  usecols=["YEAR", "OCCSOC", "PERWT", "SEX", "AGE", "EDUCD"])
print(acs.columns.tolist())
print(acs[acs["OCCSOC"] != "0"]["OCCSOC"].value_counts().head(10))

['YEAR', 'PERWT', 'SEX', 'AGE', 'EDUCD', 'OCCSOC']
OCCSOC
     0    6759002
1191XX     275782
533030     220493
412010     209963
291141     208012
252020     200466
412031     198869
434051     169319
411011     164841
537062     159243
Name: count, dtype: int64


In [ ]:
import numpy as np

# Filter out non-workers and missing
acs_clean = acs[(acs["OCCSOC"] != "0") & (acs["OCCSOC"].notna())].copy()

# Reformat OCCSOC: "291141" → "29-1141"
acs_clean["soc_code"] = acs_clean["OCCSOC"].str.strip().str[:2] + "-" + acs_clean["OCCSOC"].str.strip().str[2:]

# Flag college educated (EDUCD >= 101 = bachelor's+)
acs_clean["college"] = (acs_clean["EDUCD"] >= 101).astype(int)

# Flag female (SEX == 2)
acs_clean["female"] = (acs_clean["SEX"] == 2).astype(int)

# Aggregate to SOC × year (weighted by PERWT)
def wavg(group, col):
    return np.average(group[col], weights=group["PERWT"])

acs_occ = acs_clean.groupby(["soc_code", "YEAR"]).apply(
    lambda g: pd.Series({
        "share_female":  wavg(g, "female"),
        "share_college": wavg(g, "college"),
        "mean_age":      wavg(g, "AGE"),
        "n_workers":     g["PERWT"].sum(),
    })
).reset_index().rename(columns={"YEAR": "year"})

print(acs_occ.shape)
print(acs_occ.head(10))

(2655, 6)
  soc_code  year  share_female  share_college   mean_age    n_workers
0       0-  2019      0.542577       0.092379  33.831433  133182103.0
1       0-  2021      0.538070       0.099211  33.756008  132434936.0
2       0-  2022      0.539691       0.103750  34.330987  130990215.0
3       0-  2023      0.538268       0.105907  34.464288  131264156.0
4       0-  2024      0.538312       0.110634  34.750822  132852907.0
5  11-1021  2019      0.345429       0.450465  44.658795    1245876.0
6  11-1021  2021      0.350218       0.448013  44.813153    1344891.0
7  11-1021  2022      0.355951       0.448428  45.071751    1427742.0
8  11-1021  2023      0.357487       0.466168  44.932418    1508976.0
9  11-1021  2024      0.373119       0.450356  45.106770    1610378.0


/tmp/ipykernel_26464/1448259646.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  acs_occ = acs_clean.groupby(["soc_code", "YEAR"]).apply(


In [ ]:
acs_occ = acs_occ[acs_occ["soc_code"] != "0-"].copy()

In [ ]:
print(acs_occ[acs_occ["soc_code"].str.contains("X")]["soc_code"].unique())
print(f"X-codes: {acs_occ['soc_code'].str.contains('X').sum()} rows")

['11-10XX' '11-91XX' '13-20XX' '15-124X' '15-20XX' '17-20XX' '17-21XX'
 '17-301X' '17-302X' '19-10XX' '19-204X' '19-303X' '19-30XX' '19-40XX'
 '21-109X' '23-10XX' '25-30XX' '25-90XX' '27-102X' '27-40XX' '29-112X'
 '29-11XX' '29-203X' '29-205X' '31-113X' '31-909X' '33-30XX' '33-909X'
 '37-201X' '37-301X' '39-30XX' '39-40XX' '39-509X' '43-30XX' '43-4XXX'
 '43-9XXX' '45-20XX' '47-2XXX' '47-40XX' '47-50XX' '49-209X' '49-904X'
 '49-90XX' '51-20XX' '51-403X' '51-40XX' '51-4XXX' '51-609X' '51-70XX'
 '51-91XX' '53-40XX' '53-50XX' '53-60XX' '53-70XX' '53-71XX']
X-codes: 275 rows


In [ ]:
focal = ["15-1252", "15-2051", "13-2051", "23-2011", "35-3023", "47-2061", "31-1120"]
for soc in focal:
    match = acs_occ[acs_occ["soc_code"] == soc]
    print(soc, "→", "FOUND" if not match.empty else "MISSING")

15-1252 → FOUND
15-2051 → MISSING
13-2051 → FOUND
23-2011 → FOUND
35-3023 → FOUND
47-2061 → FOUND
31-1120 → MISSING


In [ ]:
print(acs_occ[acs_occ["soc_code"].str.startswith("15-20")]["soc_code"].unique())
print(acs_occ[acs_occ["soc_code"].str.startswith("31-11")]["soc_code"].unique())

['15-2011' '15-2031' '15-20XX']
['31-1121' '31-1122' '31-1131' '31-113X']


In [ ]:
siblings = {
    "15-2051": ["15-2011", "15-2031"],
    "31-1120": ["31-1121", "31-1122"],
}
demo_cols = ["share_female", "share_college", "mean_age"]

for soc, sibs in siblings.items():
    for yr in acs_occ["year"].unique():
        sib_rows = acs_occ[(acs_occ["soc_code"].isin(sibs)) & (acs_occ["year"] == yr)]
        if not sib_rows.empty:
            imputed = sib_rows[demo_cols].mean()
            imputed["soc_code"] = soc
            imputed["year"] = yr
            imputed["n_workers"] = sib_rows["n_workers"].sum()
            acs_occ = pd.concat([acs_occ, pd.DataFrame([imputed])], ignore_index=True)

print(acs_occ[acs_occ["soc_code"].isin(["15-2051", "31-1120"])])

     soc_code  year  share_female  share_college   mean_age  n_workers
2650  15-2051  2019      0.445816       0.847401  42.247865   219131.0
2651  15-2051  2021      0.372471       0.843920  43.488487   236796.0
2652  15-2051  2022      0.390965       0.862441  42.521719   273386.0
2653  15-2051  2023      0.416336       0.872730  42.433491   275036.0
2654  15-2051  2024      0.434232       0.857871  42.774500   278978.0
2655  31-1120  2019      0.869491       0.110874  46.213367  2565029.0
2656  31-1120  2021      0.857533       0.118923  46.968011  2780335.0
2657  31-1120  2022      0.853436       0.120154  47.485616  2704367.0
2658  31-1120  2023      0.849993       0.130864  47.972015  2730337.0
2659  31-1120  2024      0.842573       0.134975  47.599873  2861782.0


In [ ]:
import zipfile
import os

zipfile.ZipFile("data_backup.zip").extractall(".")

import os
for f in os.listdir("data/processed"):
    print(f)

merge_diagnostics.csv
oews_exposure_merged.csv
oews_crosswalk_log.csv
oews_national_panel.csv
oews_suppression_log.csv


In [ ]:
panel = pd.read_csv("data/processed/oews_exposure_onet.csv")

merged = panel.merge(
    acs_occ[["soc_code", "year", "share_female", "share_college", "mean_age"]].rename(columns={"soc_code": "occ_code"}),
    on=["occ_code", "year"],
    how="left",
)

print(merged.shape)
print(merged[["occ_code", "year", "share_female", "share_college", "mean_age"]].head(10))
print(f"Matched: {merged['share_female'].notna().sum()} of {len(merged)} rows")

(5731, 36)
  occ_code  year  share_female  share_college   mean_age
0  11-1011  2019           NaN            NaN        NaN
1  11-1011  2020           NaN            NaN        NaN
2  11-1011  2021           NaN            NaN        NaN
3  11-1011  2022           NaN            NaN        NaN
4  11-1011  2023           NaN            NaN        NaN
5  11-1011  2024           NaN            NaN        NaN
6  11-1011  2025           NaN            NaN        NaN
7  11-1021  2019      0.345429       0.450465  44.658795
8  11-1021  2020           NaN            NaN        NaN
9  11-1021  2021      0.350218       0.448013  44.813153
Matched: 1850 of 5731 rows


In [ ]:
print(acs_occ[acs_occ["soc_code"].str.startswith("11-10")]["soc_code"].unique())

['11-1021' '11-10XX']


In [ ]:
# How many unique occ_codes matched?
matched_occs = merged[merged["share_female"].notna()]["occ_code"].nunique()
total_occs = merged["occ_code"].nunique()
print(f"Matched {matched_occs} of {total_occs} occupations")

Matched 377 of 861 occupations


In [ ]:
merged.to_csv("data/processed/oews_exposure_onet_acs.csv", index=False)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

df = pd.read_csv("data/processed/oews_exposure_onet_acs.csv")

df = df[
    (df["balanced"]              == 1) &
    (df["suppressed_wage"]       == 0) &
    (df["exposure_residual"]     == 0) &
    (df["dv_beta"].notna())            &
    (df["onet_controls_matched"] == 1) &
    (df["share_female"].notna())
].copy()

df["D"] = df["dv_beta"] * df["post"]

CONTROL_VARS = [
    "pct_bachelors_plus", "routine_cognitive_index", "routine_manual_index",
    "nonroutine_analytic_index", "nonroutine_interpersonal_index",
    "routine_task_intensity", "share_female", "share_college", "mean_age",
]
OUTCOMES = ["log_wage", "log_emp", "log_wbill"]

print(f"Sample: {len(df):,} rows, {df['occ_code'].nunique():,} occupations")

Sample: 1,625 rows, 325 occupations


In [ ]:
for outcome in OUTCOMES:
    result = double_lasso_did(df, outcome, CONTROL_VARS)
    print(result)

{'outcome': 'log_wage', 'beta': np.float64(-0.077103), 'se_hc3': np.float64(0.010917), 'se_clustered': np.float64(0.011804), 'tstat': np.float64(-7.0628), 'pval': np.float64(0.0), 'ci_low_95': np.float64(-0.098499), 'ci_high_95': np.float64(-0.055706), 'n_obs': 1625, 'n_occ': 325, 'n_years': 5, 'alpha_y': np.float64(0.00208506), 'alpha_d': np.float64(0.00429587), 'selected_in_y': [], 'selected_in_d': ['share_college'], 'selected_either': ['share_college']}
{'outcome': 'log_emp', 'beta': np.float64(-0.024497), 'se_hc3': np.float64(0.019249), 'se_clustered': np.float64(0.029757), 'tstat': np.float64(-1.2726), 'pval': np.float64(0.2031), 'ci_low_95': np.float64(-0.062224), 'ci_high_95': np.float64(0.01323), 'n_obs': 1625, 'n_occ': 325, 'n_years': 5, 'alpha_y': np.float64(1.179e-05), 'alpha_d': np.float64(0.00429587), 'selected_in_y': ['share_female', 'share_college', 'mean_age'], 'selected_in_d': ['share_college'], 'selected_either': ['share_female', 'share_college', 'mean_age']}
{'outcom

In [ ]:
import pandas as pd
import os

SAVE_PATH = "results.csv"

# load existing completed outcomes
if os.path.exists(SAVE_PATH):
    existing = pd.read_csv(SAVE_PATH)
    completed = set(existing["outcome"])
else:
    completed = set()

for outcome in OUTCOMES:

    if outcome in completed:
        continue

    result = double_lasso_did(df, outcome, CONTROL_VARS)

    # add outcome name into result dict
    result["outcome"] = outcome

    row = pd.DataFrame([result])

    # append immediately so crashes don't lose progress
    if os.path.exists(SAVE_PATH):
        row.to_csv(SAVE_PATH, mode="a", header=False, index=False)
    else:
        row.to_csv(SAVE_PATH, index=False)

    print(f"Saved {outcome}")

Saved log_wage
Saved log_emp
Saved log_wbill


### Validate Parallel Trends Assumption

In [ ]:
import pandas as pd
import zipfile

zipfile.ZipFile("data_backup.zip").extractall(".")

In [3]:
df = pd.read_csv("oews_exposure_onet.csv")
df = df[
    (df["balanced"]              == 1) &
    (df["suppressed_wage"]       == 0) &
    (df["exposure_residual"]     == 0) &
    (df["dv_beta"].notna())            &
    (df["onet_controls_matched"] == 1)
].copy()

print(f"{len(df):,} rows, {df['occ_code'].nunique():,} occupations")

4,871 rows, 696 occupations


In [16]:
sub = df.dropna(subset=["log_wage"] + CONTROL_VARS_ONET + ["D"]).copy().reset_index(drop=True)
sub["et"] = sub["year"] - 2022

SELECTED_CONTROLS = ["routine_cognitive_index", "routine_task_intensity"]

# Residualize log_wage on FEs + selected controls
controls_str = " + ".join(SELECTED_CONTROLS)
sub["v_tilde"] = smf.ols(
    f"log_wage ~ C(occ_code) + C(year) + {controls_str}",
    data=sub
).fit().resid

# Residualize dv_beta on selected controls (cross-sectional, one row per occ)
occ_df = sub.groupby("occ_code")[["dv_beta"] + SELECTED_CONTROLS].first().reset_index()
X_occ = np.column_stack([np.ones(len(occ_df)), occ_df[SELECTED_CONTROLS].values])
u_occ = occ_df["dv_beta"].values - X_occ @ np.linalg.lstsq(X_occ, occ_df["dv_beta"].values, rcond=None)[0]
occ_df["u_tilde"] = u_occ
sub = sub.merge(occ_df[["occ_code", "u_tilde"]], on="occ_code", how="left")

# Event study
for t in [-3, -2, -1, 1, 2, 3]:
    col = f"d_et_m{abs(t)}" if t < 0 else f"d_et_p{t}"
    sub[col] = (sub["et"] == t).astype(int) * sub["u_tilde"]

interaction_terms = " + ".join(
    [f"d_et_m{abs(t)}" for t in [-3, -2, -1]] +
    [f"d_et_p{t}" for t in [1, 2, 3]]
)
res = smf.ols(
    f"v_tilde ~ {interaction_terms} + C(occ_code) + C(year)",
    data=sub
).fit(cov_type="HC3")

coefs, ci_low, ci_high, pvals = {}, {}, {}, {}
for t in [-3, -2, -1, 1, 2, 3]:
    col = f"d_et_m{abs(t)}" if t < 0 else f"d_et_p{t}"
    coefs[t]  = res.params[col]
    ci_low[t] = res.conf_int().loc[col][0]
    ci_high[t] = res.conf_int().loc[col][1]
    pvals[t]  = res.pvalues[col]

print("\nConditional parallel trends test (log_wage residuals):")
print(pd.DataFrame({"coef": coefs, "ci_low": ci_low, "ci_high": ci_high, "pval": pvals}).round(4))


Conditional parallel trends test (log_wage residuals):
      coef  ci_low  ci_high    pval
-3  0.0100 -0.0411   0.0611  0.7005
-2  0.0191 -0.0301   0.0682  0.4471
-1 -0.0160 -0.0615   0.0294  0.4895
 1 -0.0326 -0.0791   0.0138  0.1686
 2 -0.0316 -0.0756   0.0125  0.1607
 3 -0.0248 -0.0680   0.0184  0.2603


In [ ]:
# Robustness: restrict to 2021-2024 (tighter pre-period, flat pre-trends)
df_restricted = df[df["year"] >= 2021].copy()
print(f"Restricted sample: {df_restricted['year'].unique()}")

for outcome in OUTCOMES:
    result = double_lasso_did(df_restricted, outcome, CONTROL_VARS_ONET)
    result["spec"] = "restricted_2021_2024"
    print(result)

Restricted sample: [2021 2022 2023 2024 2025]
{'outcome': 'log_wage', 'beta': np.float64(-0.049551), 'se_hc3': np.float64(0.007043), 'se_clustered': np.float64(0.00667), 'tstat': np.float64(-7.0359), 'pval': np.float64(0.0), 'ci_low_95': np.float64(-0.063354), 'ci_high_95': np.float64(-0.035748), 'n_obs': 3479, 'n_occ': 696, 'n_years': 5, 'alpha_y': np.float64(0.00169081), 'alpha_d': np.float64(0.00298397), 'selected_in_y': [], 'selected_in_d': ['routine_manual_index', 'nonroutine_interpersonal_index', 'routine_task_intensity'], 'selected_either': ['routine_manual_index', 'nonroutine_interpersonal_index', 'routine_task_intensity'], 'spec': 'restricted_2021_2024'}
{'outcome': 'log_emp', 'beta': np.float64(0.021509), 'se_hc3': np.float64(0.015694), 'se_clustered': np.float64(0.024993), 'tstat': np.float64(1.3705), 'pval': np.float64(0.1705), 'ci_low_95': np.float64(-0.009251), 'ci_high_95': np.float64(0.052269), 'n_obs': 3479, 'n_occ': 696, 'n_years': 5, 'alpha_y': np.float64(0.00333373)

In [ ]:
!zip -r data_backup_final.zip data/processed/ results/

  adding: data/processed/ (stored 0%)
  adding: data/processed/merge_diagnostics.csv (deflated 74%)
  adding: data/processed/oews_exposure_merged.csv (deflated 78%)
  adding: data/processed/oews_crosswalk_log.csv (deflated 52%)
  adding: data/processed/oews_national_panel.csv (deflated 68%)
  adding: data/processed/oews_suppression_log.csv (deflated 68%)
  adding: data/processed/oews_exposure_onet.csv (deflated 81%)
  adding: results/ (stored 0%)
  adding: results/double_lasso_selected_vars.csv (deflated 76%)
  adding: results/double_lasso_results.csv (deflated 55%)
  adding: results/double_lasso_restricted.csv (deflated 69%)
  adding: results/coef_plot.png (deflated 19%)
  adding: results/double_lasso_all_specs.csv (deflated 47%)


In [ ]:
CONTROL_VARS_ONET = [
    "pct_bachelors_plus", "routine_cognitive_index", "routine_manual_index",
    "nonroutine_analytic_index", "nonroutine_interpersonal_index", "routine_task_intensity",
]

df["D"] = df["dv_beta"] * df["post"]
df_restricted = df[df["year"] >= 2021].copy()

restricted_results = []
for outcome in ["log_wage", "log_emp", "log_wbill"]:
    result = double_lasso_did(df_restricted, outcome, CONTROL_VARS_ONET)
    result["spec"] = "restricted_2021_2025"
    restricted_results.append(result)

pd.DataFrame(restricted_results).to_csv("results/double_lasso_restricted.csv", index=False)
print("Saved")

Saved


In [ ]:
import numpy as np
import pandas as pd
import zipfile
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import statsmodels.formula.api as smf